<a href="https://colab.research.google.com/github/mdondzik/air-quality-interpolation/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection – Warsaw PM2.5

This notebook focuses on collecting real-world air quality measurements for Warsaw. The data is sourced from the OpenAQ platform, which aggregates environmental sensor readings from multiple monitoring stations.  

The goal is to obtain a **clean, reproducible dataset** of avarage PM2.5 concentrations across the city in winter months (december 2024-april 2025), suitable for spatial analysis and interpolation. Steps include:

- Retrieving measurements for all Warsaw stations within a selected time period  
- Filtering invalid or extreme values  
- Aggregating multiple measurements per location to produce a single representative value  

This dataset forms the foundation for the subsequent analysis, visualization, and modeling in this project.

Finding all Warsaw (and neighbouring) stations:

In [1]:
!pip install openaq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
from openaq import OpenAQ

api = OpenAQ(api_key="4aa2b8f0762a835f4d664afb82b1da54afaee6093393628252ff31f87309e51a")

locations = api.locations.list(
    bbox=(20.8, 52.0, 21.3, 52.4),  # Warsaw bounds
    parameters_id=2,
    limit=1000
)

print(len(locations.results))

24


Choosing sensors measuring PM2.5:

In [3]:
sensor_data = []

for loc in locations.results:
    for sensor in loc.sensors:
        if sensor.parameter.id == 2:  # PM2.5
            sensor_data.append({
                "sensor_id": sensor.id,
                "lat": loc.coordinates.latitude,
                "lon": loc.coordinates.longitude
            })

print(f"Found {len(sensor_data)} PM2.5 sensors")

Found 25 PM2.5 sensors


Fetching monthly avarage PM2.5 for each sensor:

In [4]:
all_data = []

for s in sensor_data:

    measurements = api.measurements.list(
        sensors_id=s["sensor_id"],
        data="days",
        rollup="monthly",
        limit=1000
    )

Filtering data by date: